# Mimakiwa Cloud Training

Trains your Mimakiwa model on Colab's free CPU runtime (NdArray backend — no GPU drivers needed).

**Steps:**
1. Run all cells top to bottom
2. Wait for training (~45–90 min on Colab CPU for 20k steps)
3. Download the 3 output files at the end
4. On your Mac: `rm -rf ~/.mimakiwa/ && mkdir ~/.mimakiwa`
5. Copy the 3 downloaded files into `~/.mimakiwa/`
6. `cargo run --release` — app loads trained model, goes straight to chat

> **Tip:** Go to Runtime → Change runtime type → GPU (T4) for ~4x faster training

In [ ]:
# ── Cell 1: Install Rust ──────────────────────────────────────────────────────
import subprocess, os

result = subprocess.run(
    'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else result.stderr[-2000:])

# Add cargo to PATH for this session
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
print('Rust version:', subprocess.check_output('rustc --version', shell=True).decode().strip())

In [ ]:
# ── Cell 2: Clone repo ────────────────────────────────────────────────────────
import subprocess

REPO_URL = 'https://github.com/aryansrao/mimakiwa.git'

r = subprocess.run(f'git clone {REPO_URL} mimakiwa', shell=True, capture_output=True, text=True)
print(r.stdout or r.stderr)

# If already cloned, just pull latest
r2 = subprocess.run('cd mimakiwa && git pull', shell=True, capture_output=True, text=True)
print(r2.stdout or r2.stderr)

In [ ]:
# ── Cell 3: Set up LibTorch ───────────────────────────────────────────────────
import subprocess, os

libtorch = subprocess.check_output(
    "python3 -c \"import torch; print(torch.__path__[0])\"", shell=True
).decode().strip()

os.environ['LIBTORCH']                    = libtorch
os.environ['LIBTORCH_USE_PYTORCH']        = '1'
os.environ['LIBTORCH_BYPASS_VERSION_CHECK'] = '1'   # tch expects 2.2, Colab has 2.11 — safe to bypass
os.environ['LD_LIBRARY_PATH']             = f"{libtorch}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['PATH']                        = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

print(f"LIBTORCH={libtorch}")
print("PyTorch:", subprocess.check_output(
    "python3 -c \"import torch; print(torch.__version__)\"", shell=True
).decode().strip())

In [ ]:
# ── Cell 4: Build with CUDA (one-time, ~15–25 min) ───────────────────────────
import subprocess, time, os

t0 = time.time()
print('Building cloud-train with CUDA backend...')

env = os.environ.copy()
r = subprocess.run(
    'cd mimakiwa && cargo build --release -p mimakiwa-cloud-train --features cuda 2>&1',
    shell=True, capture_output=True, text=True, env=env
)
print(r.stdout[-4000:] if r.stdout else r.stderr[-4000:])
print(f'Build finished in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Cell 5: Train on T4 GPU ───────────────────────────────────────────────────
import subprocess, os

os.makedirs('/content/model_out', exist_ok=True)

# Dolly-15k: real instruction-following Q&A dataset (not children's stories)
DATASET_URL = 'https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl'
MODEL  = 'small'   # 'compact' (~15M) or 'small' (~85M, better)
STEPS  = 20000
MAX_MB = 5         # Dolly is only 3 MB

cmd = (
    f'mimakiwa/target/release/cloud-train '
    f'--model {MODEL} '
    f'--dataset-url "{DATASET_URL}" '
    f'--steps {STEPS} --max-mb {MAX_MB} '
    f'--out /content/model_out'
)

print(f'Training {MODEL} model on T4 GPU | Dolly-15k dataset (real Q&A)')
print('─' * 60)

proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True,
                        env=os.environ.copy())
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('─' * 60)
print('Return code:', proc.returncode)

## After downloading

On your Mac, run:
```bash
rm -rf ~/.mimakiwa
mkdir ~/.mimakiwa
# Move the 3 downloaded files into ~/.mimakiwa/
mv ~/Downloads/mimakiwa.bin       ~/.mimakiwa/
mv ~/Downloads/mimakiwa.cfg.json  ~/.mimakiwa/
mv ~/Downloads/tokenizer.json     ~/.mimakiwa/

# Launch app — goes straight to Chat, no retraining
cargo run --release
```